# RCC terminal-state cost bounds

Compute a conditional preparation-cost lower bound from a complete exact spectrum or fixed-projector counts. Both routes use the installed `rcc_refcert` package.

Follow the [notebook setup instructions](README.md#jupyter-notebook) to install this source tree, register its kernel, and launch Jupyter. If you already manage a Python environment, run these commands from the checkout with that environment's interpreter:

```bash
python -m pip install ".[notebook]"
python -m ipykernel install --user --name rcc-refcert --display-name "Python (rcc-refcert)"
python -m jupyterlab
```

Select **Python (rcc-refcert)** as the kernel. Installing notebook dependencies alone does not register that name. A downloaded notebook also works with an environment where this version of the package and its notebook dependencies are installed and that kernel is registered.

The example is one fixed four-dimensional model with eight terminal outputs. Every named target has a one-slot construction; this small model illustrates the calculation and does not establish scaling for a model family.


In [ ]:
import json
from copy import deepcopy
from pathlib import Path

import rcc_refcert
from rcc_refcert.bounds import (
    InputError,
    bound_from_counts,
    bound_from_spectrum,
    load_template,
    prepare_protocol,
    render_record,
    replay_record,
)

print("rcc-refcert", rcc_refcert.__version__)

## Exact spectrum

The template declares the uniform reference on $d_R=4$, eight distinct atomic action classes, exact transcription and the fixed model's reference-domination argument. Those are external premises of the calculator. Change the task and its supporting sources when using a different model.

Supply all eigenvalues, including zeros, as exact strings, integers, `Fraction` or `Decimal`. The declared spectrum must sum to one exactly.

In [ ]:
request = load_template("spectrum")
result = bound_from_spectrum(request)
print(render_record(result))
assert result["confidence"] is None
assert result["cost"]["integer_lower_bound_slots"] == 1

## Tolerance and a valid zero result

Increase the normalized trace-distance tolerance. At $\epsilon=0.35$, the example mixed state can be smoothed to its uniform reference. A zero lower bound means this calculation no longer excludes low cost; it does not establish zero preparation cost.

In [ ]:
for epsilon in ("0", "0.1", "0.3", "0.35", "1"):
    changed = deepcopy(request)
    changed["task"]["target"]["epsilon"] = epsilon
    record = bound_from_spectrum(changed)
    print(
        epsilon, record["display"]["lower_bound_slots"], record["cost"]["zero_reason"]
    )
assert record["cost"]["lower_bound_slots"] == "0"

## A named pure target

The reference dimension stays four when the target has rank one. The continuous bound is $2/3$ of an atomic slot; the declared integer-slot cost domain strengthens it to one slot.

In [ ]:
pure = deepcopy(request)
pure["task"]["target"].update(id="pure-0", epsilon="0")
pure["spectrum"] = [1, 0, 0, 0]
pure_result = bound_from_spectrum(pure)
assert pure_result["cost"]["integer_lower_bound_slots"] == 1
print(render_record(pure_result))

## Declare a protocol before acquiring counts

Freeze the target, reference, projector, rank, fixed sample size and failure budget before acquisition. Save the full protocol and its digest with the data. The digest checks identity; it cannot attest chronology or iid sampling.

In [ ]:
specification = load_template("protocol")
frozen = prepare_protocol(specification["task"], specification["protocol"])
print(json.dumps(frozen, indent=2))

## Fixed-projector counts

These 550 hits out of 1000 are an explicitly synthetic illustration, not experimental evidence. The one-sided Hoeffding event has coverage at least $1-\delta=0.95$ under the declared fixed-N iid sampling model. That coverage is separate from the truth of the physical model premises. Real data must carry the digest of its original acquisition protocol.

In [ ]:
counts = load_template("counts")
assert counts["data"]["origin"] == "synthetic_fixture"
assert counts["data"]["protocol_sha256"] == frozen["protocol_sha256"]
count_result = bound_from_counts(counts)
assert count_result["confidence"]["coverage_at_least_exact"] == "19/20"
print(render_record(count_result))

## Detect a changed witness

An existing data record cannot be attached to a different projector. Correct the source of the mismatch; do not recompute its digest to retrofit the data. This initial protocol format also binds the generation tolerance.

In [ ]:
changed_counts = deepcopy(counts)
changed_counts["protocol"]["witness_id"] = "different-projector"
try:
    bound_from_counts(changed_counts)
except InputError as error:
    print("Expected rejection:", error)
else:
    raise AssertionError("Changed witness was accepted")

## Save and replay

The record contains exact input snapshots and the producer version. Replay recomputes the same-version scalar calculation and checks every field. It does not verify external model proofs or measurement data. The upper endpoint of the calculator enclosure is not an upper bound on the optimum preparation cost.

In [ ]:
output = Path("generated_results")
output.mkdir(exist_ok=True)
record_path = output / "terminal_bound.json"
record_path.write_text(json.dumps(result, indent=2) + "\n", encoding="utf-8")
replayed = replay_record(json.loads(record_path.read_text(encoding="utf-8")))
assert replayed["matches"]
print(replayed)